In [4]:
import pandas as pd
from openai import OpenAI
from tqdm import tqdm
import os

# 1. 1단계에서 만든 통합 데이터 불러오기 및 5,000개 랜덤 샘플링
df_all = pd.read_csv('final_merged_reviews_all.csv')
sample_df = df_all.sample(n=5000, random_state=42).copy()

# 2. 이어하기(체크포인트) 세팅
checkpoint_file = 'labeled_reviews_checkpoint.csv'

if os.path.exists(checkpoint_file):
    print(f"✅ 체크포인트 발견! 이어서 진행합니다.")
    labeled_df = pd.read_csv(checkpoint_file)
    labeled_data = labeled_df.to_dict('records')
    # 이미 완료한 개수만큼 건너뛰기
    remaining_df = sample_df.iloc[len(labeled_data):].copy()
else:
    print("⚠️ 체크포인트 파일이 없습니다. 처음부터 시작합니다.")
    labeled_data = []
    remaining_df = sample_df

# 3. LM Studio 서버 연결 (LM Studio가 켜져 있고 Gemma 모델이 로드되어 있어야 합니다)
client = OpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio" 
)

def get_label_from_gemma(movie_title, text):
    # 기존에 작성하신 완벽한 룰과 예시에 '영화 제목' 문맥과 '리뷰' 단어만 최적화했습니다.
    prompt = f"""너는 영화/드라마 스포일러를 완벽하게 차단하기 위해 훈련된 '초정밀 스포일러 판독 AI'야.
오직 '1' 또는 '0'이라는 숫자 하나만 출력해야 해. 다른 부가 설명이나 기호는 절대 허용하지 않아.

[스포일러 판단 기준: 다음 중 하나라도 해당하면 무조건 1(스포일러) 출력]
1. 내용 유출: 결말, 반전, 흑막, 진짜 범인의 정체를 언급하는 경우.
2. 캐릭터 생사: 특정 등장인물의 생사(죽음, 생존)나 하차를 암시하는 경우.
3. 구체적 씬 묘사: "마지막에 둘이 떨어질 때", "차 폭발할 때" 등 영화의 특정 장면, 액션, 전개를 구체적으로 묘사하는 경우.
4. 원작 스포일러: 소설/웹툰 원작의 내용을 바탕으로 앞으로의 전개를 미리 알려주는 경우.

[안전한 리뷰 기준: 아래의 경우 0(안전) 출력]
1. 단순 감상: 기대감, 재미, 실망, 슬픔 등 내용을 유출하지 않은 단순 감상평.
2. 배우/감독 언급: 연기력, 외모, 연출에 대한 칭찬이나 비판.
3. 관람 경험담: 영화관에서의 개인적인 경험.
4. 단순 질문: "결말 어떻게 되나요?", "범인 누굴까?" 처럼 내용 유출 없이 궁금증만 표하는 경우.

[판독 예시]
리뷰: "마지막에 순간적으로 둘이 떨어질 때 옆에서 내 쌍둥이는 팝콘통 뜯었는데 ㅋㅋ" -> 1 
리뷰: "와 범인이 저 경찰일 줄은 상상도 못함 소름" -> 1 
리뷰: "원작 웹툰 다 봤는데 저기서 남주인공 기억 잃음" -> 1 
리뷰: "결말이 너무 허무해서 돈 아까웠어요 ㅠㅠ" -> 0 
리뷰: "이거 진짜 재밌었어요 다들 꼭 보세요" -> 0 
리뷰: "윤아 첨으로 연기 좋다 느낀 작품 ㅋㅋ" -> 0 

위 지침을 완벽히 숙지했다면, 아래 입력된 영화 정보를 바탕으로 리뷰를 분석하고 오직 숫자 '1' 또는 '0'만 출력해.

[대상 영화]: {movie_title}
리뷰: "{text}"
정답: """
    
    try:
        response = client.chat.completions.create(
            model="local-model",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0
        )
        
        result = str(response.choices[0].message.content).strip()
        
        if "1" in result: return 1
        elif "0" in result: return 0
        else: return 0
            
    except Exception as e:
        print(f"\n[오류 발생]: {e}")
        return 0

# 4. 라벨링 시작
print(f"🚀 남은 {len(remaining_df)}건 본격 라벨링 시작...")

for idx, row in tqdm(remaining_df.iterrows(), total=len(remaining_df)):
    movie_title = row['movie_title']
    text = row['text']
    label = get_label_from_gemma(movie_title, text)
    
    labeled_data.append({
        'movie_title': movie_title,
        'text': text,
        'is_spoiler': label
    })
    
    # 500건 단위 임시 저장
    if len(labeled_data) % 500 == 0:
        temp_df = pd.DataFrame(labeled_data)
        temp_df.to_csv(checkpoint_file, index=False, encoding='utf-8-sig')

# 5. 최종 완성본 저장
final_df = pd.DataFrame(labeled_data)
final_df.to_csv('labeled_reviews_gemma_5000_raw.csv', index=False, encoding='utf-8-sig')
print("✅ 5,000건 라벨링 1차 완료!")

⚠️ 체크포인트 파일이 없습니다. 처음부터 시작합니다.
🚀 남은 5000건 본격 라벨링 시작...


100%|██████████| 5000/5000 [19:41:11<00:00, 14.17s/it]   

✅ 5,000건 라벨링 1차 완료!


In [6]:
import pandas as pd

# 1. Gemma가 추출한 5,000건 원본 파일 불러오기
print("데이터를 불러오는 중입니다...")
df = pd.read_csv('labeled_reviews_gemma_5000_raw.csv')

# 2. 규칙 기반 자동 라벨 수정 (컴퓨터가 1차로 오답 청소)
# 이 단어들이 들어갔는데 Gemma가 안전(0)이라고 했으면 스포일러(1)로 강제 변경
spoiler_keywords = ['결말', '반전', '범인은', '마지막에 죽', '알고보니', '주인공이 죽', '스포', '쿠키영상']
# 이 단어들이 들어갔고 문장이 짧은데 Gemma가 스포(1)라고 했으면 안전(0)으로 강제 변경
safe_keywords = ['재밌다', '꿀잼', '노잼', '감동', '최고', '추천해요', '배우들 연기', '돈 아깝']

print("규칙 기반으로 자동 정제 중...")
corrected_count = 0

for idx, row in df.iterrows():
    text = str(row['text'])
    if any(k in text for k in spoiler_keywords) and row['is_spoiler'] == 0:
        df.at[idx, 'is_spoiler'] = 1
        corrected_count += 1
    elif any(k in text for k in safe_keywords) and len(text) < 30 and row['is_spoiler'] == 1:
        df.at[idx, 'is_spoiler'] = 0
        corrected_count += 1

print(f"-> 총 {corrected_count}개의 데이터가 규칙에 의해 자동 수정되었습니다.")

# 3. 최종 시험지용 200건 추출 및 학습용 4,800건 분리
# 500개를 다 보지 않고, 확실한 기준을 가진 200건만 랜덤으로 뽑아 최종 시험지로 확정합니다.
test_set = df.sample(n=200, random_state=42).reset_index(drop=True)
train_base_set = df.drop(test_set.index).reset_index(drop=True)

# 파일로 각각 저장
test_set.to_csv('test_set_clean_200.csv', index=False, encoding='utf-8-sig')
train_base_set.to_csv('train_base_4800.csv', index=False, encoding='utf-8-sig')

print("\n✅ 2-2단계 성공!")
print("- 최종 평가용 깨끗한 시험지: 'test_set_clean_200.csv' (200건)")
print("- 1차 모델 학습용 기본 데이터: 'train_base_4800.csv' (4,800건)")

데이터를 불러오는 중입니다...
규칙 기반으로 자동 정제 중...
-> 총 161개의 데이터가 규칙에 의해 자동 수정되었습니다.

✅ 2-2단계 성공!
- 최종 평가용 깨끗한 시험지: 'test_set_clean_200.csv' (200건)
- 1차 모델 학습용 기본 데이터: 'train_base_4800.csv' (4,800건)
